# Bank Marketing: Sensitive Balancing Preprocessing

Author: Ilse Harmers \
Last modified: February 24, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import utils

## Pre-Processing Bank Marketing Dataset

In [ ]:
# Importing Bank Marketing dataset (downloaded from https://archive.ics.uci.edu/dataset/222/bank+marketing on June 1, 2025).

# Assigning file path for importing the dataset. 
path = "../bank-marketing/bank-full.csv"

# Importing dataset as pandas DataFrame.
bank = pd.read_csv(path, sep = ";")
dataset_name = "Bank"
column_names = list(bank.columns)
bank

In [ ]:
# Replacing "unknown" in the dataset with NaN values.
bank.replace("unknown", np.nan, inplace=True)

In [ ]:
# Print names of columns in the dataset as a check.
print(f"{dataset_name} contains {len(column_names)} columns: \n {column_names} \n")

# Check dtypes of dataset's columns.
print("Columns' dtypes:\n", bank.dtypes)
print()

# Count missing values (reported as NaNs) in each column.
missing_values = np.array([bank[col].isna().sum() for col in column_names])

# This if-elif statement checks whether the columns in the dataset contain missing values that are reported as NaNs. 
if missing_values.any() == False:   # no non-zero entries
    print(f"{dataset_name} has no missing values in any of its columns.")
elif missing_values.any() == True:   # at least one non-zero entry (indicating the presence of missing values)
    print(f"There are {np.sum(missing_values)} missing values in {dataset_name}'s columns.")   
    for i in range(len(missing_values)):
        print(f"\t{column_names[i]}: {missing_values[i]} values")

In [ ]:
# Statistical summary of the dataset.
bank.describe()

### Removing Missing Values & Dropping Columns

Since many cells in the Bank dataset are encoded with *unknown*, we will *not* remove the rows containing missing values. 

In [ ]:
# Replacing NaN values in the dataset with "unknown".
bank.replace(np.nan, "unknown", inplace = True)

In [ ]:
# Count missing values (reported as NaNs) in each column.
missing_values = np.array([bank[col].isna().sum() for col in column_names])

# This if-elif statement checks whether the columns in the dataset contain missing values that are reported as NaNs. 
if missing_values.any() == False:   # no non-zero entries
    print(f"{dataset_name} has no missing values in any of its columns.")
elif missing_values.any() == True:   # at least one non-zero entry (indicating the presence of missing values)
    print(f"There are {np.sum(missing_values)} missing values in {dataset_name}'s columns.")   
    for i in range(len(missing_values)):
        print(f"\t{column_names[i]}: {missing_values[i]} values")

## Sensitive Balancing Preprocessing

In [ ]:
# Splitting the Bank dataset into a train and test set with a 80:20 ratio.
bank_train, bank_test = train_test_split(bank, train_size = 0.80, random_state = 42)

In [ ]:
def sens_preprocessing(df, age_thr = 25):
    """This function carries out the sensitive balancing preprocessing by randomly undersampling the demographic groups such that each group 
    has the same size. The demographic groups for Bank are 25<=-'no' (Young0), 25<=-'yes' (Young1), >25-'no' (Old0) and >25-'yes' (Old1). 
    For Bank, Young1 is the smallest group. So, Young0, Old0 and Old1 will be undersampled in order to have the same size as Young1."""
    
    young = df[df["age"] <= age_thr].shape[0]
    print(f"young (<= {age_thr}): ", young)
    old = df[df["age"] > age_thr].shape[0]
    print(f"old (> {age_thr}): ", old)

    # Creating a new column in the DataFrame with the demographic group labels.
    df.loc[df.loc[(df["age"] <= age_thr) & (df["y"] == "no")].index, "age-y"] = "Young0"
    df.loc[df.loc[(df["age"] <= age_thr) & (df["y"] == "yes")].index, "age-y"] = "Young1"
    df.loc[df.loc[(df["age"] > age_thr) & (df["y"] == "no")].index, "age-y"] = "Old0"
    df.loc[df.loc[(df["age"] > age_thr) & (df["y"] == "yes")].index, "age-y"] = "Old1"
    y1_rows = df[df["age-y"] == "Young1"].shape[0]
    print("Rows of Young1: ", y1_rows)

    print("Y0 & Y1: ", df[df["age-y"] == "Young0"].shape[0], df[df["age-y"] == "Young1"].shape[0])
    print("O0 & O1: ", df[df["age-y"] == "Old0"].shape[0], df[df["age-y"] == "Old1"].shape[0])

    # Plotting the label distribution of the demographic groups before preprocessing.
    utils.countplot(data = df, column_name = "age-y", order = ["Old0", "Old1", "Young0", "Young1"], fig_size = (8, 5))

    # Undersampling the demographic groups such that all demographic groups have the same size. 
    df_bal = pd.concat([df[df["age-y"] == "Old0"].sample(y1_rows, random_state = 42),
                       df[df["age-y"] == "Old1"].sample(y1_rows, random_state = 42),
                       df[df["age-y"] == "Young0"].sample(y1_rows, random_state = 42),
                       df[df["age-y"] == "Young1"]], axis = 0)

    # Plotting the label distribution of the demographic groups after preprocessing.
    utils.countplot(data = df_bal, column_name = "age-y", order = ["Old0", "Old1", "Young0", "Young1"], fig_size = (8, 5))

    # Resetting the DataFrame.
    df_bal = df_bal.drop(columns = ["age-y"])
    df = df.drop(columns = ["age-y"])

    return df_bal

In [ ]:
# Preprocessing the train and test set.
train_bal = sens_preprocessing(bank_train)
test_bal = sens_preprocessing(bank_test)

In [ ]:
# Statistical summary of the train set.
print(train_bal.describe())

# Saving train and test splits to csv files.
#test_bal.to_csv("./train-test-datasets/bank-marketing/bank_test.csv", index = False)
#train_bal.to_csv("./train-test-datasets/bank-marketing/bank_train.csv", index = False)

In [ ]:
# Computing the demographic parity and disparate impact metrics for the train and test 'balanced' datasets.
df = train_bal #test_bal
# Encoding the sensitive and target attributes for the fairness analysis.
y_encoded = utils.one_hot_encode(df[["y"]], order = [["no", "yes"]])
age_encoded = df["age"].copy()
age_encoded.loc[age_encoded <= 25] = 1
age_encoded.loc[age_encoded > 25] = 0
train_encoded = pd.concat([age_encoded.reset_index(drop = True), y_encoded.reset_index(drop = True)], axis = 1)

print("Demographic parity:", utils.demographic_parity(df = train_encoded, s = "age", y = "y"))
print("Disparate impact:", utils.disparate_impact(df = train_encoded, s = "age", y = "y"))